In [8]:
import shutil
from pathlib import Path

# Remove all data and models to start clean
for d in ["data", "models"]:
    if Path(d).exists():
        shutil.rmtree(d)
        print(f"Removed {d}/")

print("Clean slate ready.")

Removed data/
Removed models/
Clean slate ready.


In [9]:
%cd /kaggle/working
!rm -rf to-ai-or-not-to-ai
!git clone -b dev https://github.com/gismal/to-ai-or-not-to-ai.git
%cd to-ai-or-not-to-ai

/kaggle/working
Cloning into 'to-ai-or-not-to-ai'...
remote: Enumerating objects: 774, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (247/247), done.
remote: Total 774 (delta 141), reused 238 (delta 85), pack-reused 435 (from 1)
Receiving objects: 100% (774/774), 11.59 MiB | 23.14 MiB/s, done.
Resolving deltas: 100% (344/344), done.
/kaggle/working/to-ai-or-not-to-ai


In [10]:
!pip install -q \
    torch torchvision \
    scikit-learn \
    onnx onnxruntime \
    pydantic pydantic-settings \
    mlflow scipy joblib \
    imagehash pillow \
    python-dotenv aiosqlite

In [11]:
!pip install -q onnxscript

In [12]:
import os
from pathlib import Path

os.chdir("/kaggle/working/to-ai-or-not-to-ai")
print(f"Working directory: {os.getcwd()}")

for f in sorted(Path("models").iterdir()):
    mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s} {mb:.2f} MB")

Working directory: /kaggle/working/to-ai-or-not-to-ai
  best_checkpoint.pt                  10.96 MB
  metadata.json                       0.00 MB
  model_v1.onnx                       0.31 MB
  model_v1.onnx.data                  6.09 MB


In [13]:
import copy
import json
import torch
import torch.nn as nn
import torchvision.models as models
from pathlib import Path

project_root = Path("/kaggle/working/to-ai-or-not-to-ai")

# Rebuild architecture
model = models.mobilenet_v3_small(weights=None)
in_features = model.classifier[-1].in_features
model.classifier[-1] = nn.Linear(in_features, 2)

# Load best weights
checkpoint = torch.load(
    project_root / "models" / "best_checkpoint.pt",
    map_location="cpu"
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

for f in (project_root / "models").iterdir():
    if "model_v1.onnx" in f.name and "int8" not in f.name:
        f.unlink()
        print(f"Removed: {f.name}")
        
export_copy = copy.deepcopy(model).cpu().eval()
dummy_input  = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    export_copy,
    dummy_input,
    str(project_root / "models" / "model_v1.onnx"),
    export_params       = True,
    opset_version       = 18,
    do_constant_folding = True,
    input_names         = ["input"],
    output_names        = ["output"],
    dynamic_axes        = None,
)

meta_path = project_root / "models" / "metadata.json"
meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
meta["metrics"] = {k: round(v, 4) for k, v in checkpoint["metrics"].items()}
meta_path.write_text(json.dumps(meta, indent=4))

for f in sorted((project_root / "models").iterdir()):
    mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s} {mb:.2f} MB")
print("✓ metadata.json updated")

Removed: model_v1.onnx
Removed: model_v1.onnx.data


W0825 19:21:33.277000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0825 19:21:33.278000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0825 19:21:33.280000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0825 19:21:33.282000 58 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
  best_checkpoint.pt                  10.96 MB
  metadata.json                       0.00 MB
  model_v1.onnx                       0.29 MB
  model_v1.onnx.data                  6.09 MB
✓ metadata.json updated


In [14]:
import onnx
import os
from pathlib import Path

# Merge .onnx + .onnx.data into a single file (quantizer needs this)
print("Merging external data...")
os.chdir("models")
model_onnx = onnx.load("model_v1.onnx", load_external_data=True)
onnx.save(model_onnx, "model_v1_merged.onnx", save_as_external_data=False)
os.chdir("..")
print(f"  Merged: {Path('models/model_v1_merged.onnx').stat().st_size/1e6:.1f} MB")


Merging external data...
  Merged: 6.3 MB


In [15]:
!find /kaggle/input -maxdepth 2

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/birdy654


In [16]:
!ls /kaggle/input/datasets/birdy654

cifake-real-and-ai-generated-synthetic-images


In [17]:
import os
from pathlib import Path

CIFAKE = Path("/kaggle/input/datasets/birdy654")

# Find the actual CIFAKE root (handle varying Kaggle mount paths)
for candidate in CIFAKE.rglob("train"):
    if (candidate / "REAL").exists() and (candidate / "FAKE").exists():
        CIFAKE_ROOT = candidate.parent
        break

print(f"CIFAKE found at: {CIFAKE_ROOT}")
print(f"  train/REAL: {len(list((CIFAKE_ROOT/'train'/'REAL').iterdir())):,}")
print(f"  train/FAKE: {len(list((CIFAKE_ROOT/'train'/'FAKE').iterdir())):,}")
print(f"  test/REAL:  {len(list((CIFAKE_ROOT/'test'/'REAL').iterdir())):,}")
print(f"  test/FAKE:  {len(list((CIFAKE_ROOT/'test'/'FAKE').iterdir())):,}")

CIFAKE found at: /kaggle/input/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
  train/REAL: 50,000
  train/FAKE: 50,000
  test/REAL:  10,000
  test/FAKE:  10,000


In [ ]:
import os
import shutil
from pathlib import Path

if Path("data").exists():
    shutil.rmtree("data")

os.makedirs("data/train", exist_ok=True)
os.makedirs("data/val", exist_ok=True)

links = {
    "data/train/REAL":         CIFAKE_ROOT / "train" / "REAL",
    "data/train/AI_GENERATED": CIFAKE_ROOT / "train" / "FAKE",
    "data/val/REAL":           CIFAKE_ROOT / "test"  / "REAL",
    "data/val/AI_GENERATED":   CIFAKE_ROOT / "test"  / "FAKE",
}

for dst, src in links.items():
    shutil.copytree(src, dst, dirs_exist_ok=True)
    count = len(list(Path(src).iterdir()))
    print(f"✓ {dst:35s} → {count:,} images")

✓ data/train/REAL                     → 50,000 images


In [ ]:
# MLflow deprecated file-based tracking to prevent concurrency issues in production.
# However, since we are running a single isolated process in Kaggle, 
# forcing file storage is perfectly safe and avoids unnecessary database setup.
!PYTHONPATH=. MLFLOW_ALLOW_FILE_STORE=true python scripts/train.py

In [ ]:
import json
from pathlib import Path

print("=== Files saved ===")
for f in sorted(Path("models").iterdir()):
    mb = f.stat().st_size / 1_048_576
    print(f"  {f.name:30s} {mb:.1f} MB")

print("\n=== Best metrics ===")
meta = json.loads(Path("models/metadata.json").read_text())
for k, v in meta["metrics"].items():
    print(f"  {k:15s} {v:.4f}")

In [ ]:
import json
from pathlib import Path

meta = json.loads(Path("models/metadata.json").read_text())
print("=== Training results ===")
for k, v in meta["metrics"].items():
    print(f"  {k:15s} {v:.4f}")

assert meta["metrics"]["accuracy"] > 0.85, \
    f"Training failed — accuracy {meta['metrics']['accuracy']:.2f} is too low"
print("\n✓ Training looks good")

In [ ]:
from pathlib import Path

models_dir = Path("/kaggle/working/to-ai-or-not-to-ai/models")

for name in ["model_v1_merged.onnx", "model_v1_prep.onnx"]:
    f = models_dir / name
    if f.exists():
        f.unlink()
        print(f"Removed: {name}")

print("\n=== Download these files ===")
for f in sorted(models_dir.iterdir()):
    mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s} {mb:.2f} MB")

In [ ]:
import os
project_root = Path("/kaggle/working/to-ai-or-not-to-ai")
os.chdir(project_root)

!PYTHONPATH=. python scripts/calibrate.py \
    --model  models/model_v1.onnx \
    --data   data/val \
    --output models/calibrator.pkl

In [ ]:
import json
from pathlib import Path

project_root = Path("/kaggle/working/to-ai-or-not-to-ai")

print("=== All model files ===")
for f in sorted((project_root / "models").iterdir()):
    mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s} {mb:.2f} MB")

meta = json.loads((project_root / "models" / "metadata.json").read_text())

print("\n=== Training results ===")
for k, v in meta["metrics"].items():
    print(f"  {k}: {v:.4f}")

if "calibration" in meta:
    c = meta["calibration"]
    print("\n=== Calibration ===")
    print(f"  Temperature T: {c.get('temperature', 'N/A')}")
    print(f"  Brier before:  {c.get('brier_before', 'N/A')}")
    print(f"  Brier after:   {c.get('brier_after', 'N/A')}")